<a href="https://colab.research.google.com/github/stagsz/find.bi/blob/master/Kopia_av_LLM_Workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note: In order to use the GPU in google colab, click on "Runtime" -> "Change runtime type" -> select "T4 GPU" under hardware accelerator -> "save"

Install required libraries

In [ ]:
!pip install -U bitsandbytes transformers accelerate trl evaluate rouge_score &> /dev/null

In [ ]:
import os
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType
from trl import SFTTrainer
import evaluate

In [ ]:
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Note: Make sure this runs on cuda. If it does not, click on "Runtime" -> "Change runtime type" -> select "T4 GPU" under hardware accelerator -> "save"

## 1) Dataset

We load the *CARDBiomedBench* dataset from Hugging Face

You can find more information about the dataset at **[Hugging Face](https://huggingface.co/datasets/NIH-CARD/CARDBiomedBench)**, or by reading the [paper](https://www.biorxiv.org/content/10.1101/2025.01.15.633272v2.full.pdf)

In [ ]:
dataset = load_dataset("NIH-CARD/CARDBiomedBench")
dataset

The dataset contains question-answer pairs



In [ ]:
question = dataset["train"][0]["question"]
answer = dataset["train"][0]["answer"]
print("Question: ", question)
print("Answer  : ", answer)

Let's only use data samples related to category "Pharmacology"

In [ ]:
dataset = dataset.filter(lambda x: x["bio_category"] == "Pharmacology")

train_dataset = dataset["train"]
test_dataset = dataset["test"]

# for computational resons, select a smaller subset
train_dataset = train_dataset.shuffle(seed=42).select(range(1000))
test_dataset = test_dataset.shuffle(seed=42).select(range(200))

print("Num samples in train set: ", len(train_dataset))
print("Num samples in test set : ", len(test_dataset))

## 2) LLM

In this example, we use the [SmolLM2](https://huggingface.co./HuggingFaceTB/SmolLM2-135M) decoder model, developed by Hugging Face. The SmolLM2 models come in three sizes (135M, 360M, and 1.7B parameters) and are developed to solve a wide range of tasks while being lightweight enough to run on-device.
Here, we choose the 135M parameter model for computational reasons.

Let's load the model and tokenizer through Hugging Face



In [ ]:
model_name = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32).to(device)

tokenizer.pad_token = tokenizer.eos_token  # set end-of-sequence token as padding token. Used to fill blank tokens in a batch
model.config.pad_token_id = model.config.eos_token_id  # tell model which token to use for padding

In [ ]:
model.num_parameters() # 134,515,008 parameters

In [ ]:
print(f"Model size in MB: {(model.get_memory_footprint() / 1024**2):.2f}")

## 3) QLoRA

By applying QLoRA, we reduce:
- The memory footprint of the model
- The number of trainable parameters

We use the library BitsAndBytes to add quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Loads model weights in 4-bit precision. Dequantized to higher precision during computation.
    bnb_4bit_quant_type="nf4",              # the quantization data type - nf4 (NormalFloat4)
    bnb_4bit_compute_dtype=torch.bfloat16,  # The precision used for actual computations during forward/backward passes. LoRA parameters are dequantized to bfloat16 during training
)

In [ ]:
# load quantized model
model = AutoModelForCausalLM.from_pretrained(model_name,
            quantization_config=bnb_config,
            device_map="auto")

model.config.pad_token_id = model.config.eos_token_id

In [ ]:
print(f"Model size in MB: {(model.get_memory_footprint() / 1024**2):.2f}")

We reduced the memory footprint of the model, but now we want to also reduce the number of trainable parameters.
For this, we add LoRA.

In [ ]:
lora_config = LoraConfig(
    r=8,                                    # rank of low-rank decomposition matrix. Lower r leads to fewer LoRA parameters
    lora_alpha=16,                          # the higher, the more aggressive fine-tuning will be
                                            # effective learning rate: (lora_alpha/r) * learning_rate
    target_modules=["q_proj", "v_proj"],    # layers in transformer to apply LoRA to
    lora_dropout=0.1,                       # prevents overfitting
    task_type=TaskType.CAUSAL_LM,           # for autoregressive models
)


model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

By training only 0.34 % of the total parameters, we update only a small fraction of the total parameters!

Now we added quantization + LoRA = QLoRA

## 4) Get (untrained) LLM Predictions

Let's generate an example output of our base LLM.
Note that the model is not instruction-tuned (unlike ChatGPT). It is only trained to predict the next token in a sequence and is less useful for interactive tasks. It has not been trained to use an end-of-sequence token to stop answering. We need to fine-tune it for this.

In [ ]:
inputs = tokenizer("The capital of Sweden is ", return_tensors="pt", padding=True).to(device)

outputs = model.generate(inputs["input_ids"],
                         attention_mask=inputs["attention_mask"],
                         pad_token_id=tokenizer.eos_token_id,
                         max_new_tokens=50)

print(tokenizer.decode(outputs[0]))

Let's test the SmolLM2 base model on some questions from our dataset...



In [ ]:
def test_model(index, dataset, model, tokenizer):
    data = dataset[index]
    question = data["question"]

    instruction = "You are a knowledgeable assistant. Answer this question truthfully!"

    # Format the input into instruction format
    prompt = (
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{question}\n\n"
        "### Response:\n"
    )

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate response
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=100,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )

    # decode the response & remove special tokens
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # remove prompt from response
    response = response[len(prompt):]

    expected_response = data["answer"]

    return question, response, expected_response

In [ ]:
index = 0

question, response, expected_response = test_model(index, train_dataset, model, tokenizer)

print("\n")
print("question: \n", question, "\n")
print("model output: \n", response, "\n")
print("expected output: \n", expected_response)

## 5) Centralized Supervised Fine-tuning

We now want to fine-tune the model on the train dataset. For this, we convert the training data to instruction format. This is the correct format for generative question-answering tasks

In [ ]:
def generate_instruction_format(example):
    question = example["question"]
    answer = example["answer"]

    instruction = "You are a knowledgeable assistant. Answer this question truthfully!"

    prompt = (
        "### Instruction:\n"
        f"{instruction.strip()}\n\n"
        "### Input:\n"
        f"{question.strip()}\n\n"
        "### Response:\n"
        f"{answer.strip()}" + tokenizer.eos_token
    )
    return {"text": prompt}


In [ ]:
mapped_train_dataset = train_dataset.map(
    generate_instruction_format,
    remove_columns=train_dataset.column_names,
    batched=False,
)
mapped_train_dataset[0]

In [ ]:
use_cuda = torch.cuda.is_available()
print("cuda ", use_cuda)

model.train()

training_args = TrainingArguments(
    output_dir="qa-finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=5e-4,
    logging_steps=20,
    save_total_limit=2,
    use_cpu=not(use_cuda)
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=mapped_train_dataset
)

trainer.train()

Now we can check some outputs of our fine-tuned model

In [ ]:
index = 2

question, response, expected_response = test_model(index, test_dataset, model, tokenizer)

print("\n")
print("question: ", question, "\n")
print("model output: ", response, "\n")
print("expected output: ", expected_response)

## 6) Evaluation

Of course, we cannot check every output individually.

Instead, we use ROUGE-L as a metric to evaluate the fine-tuned model on the test dataset.

The **ROUGE-L** score is based on the longest common subsequence (LCS) between the generated and the reference text.
The LCS is the longest sequence of words that appear in order in both generated and reference text.
The words do **NOT** need to be contiguous.

In [ ]:
rouge = evaluate.load("rouge")

reference_text = ["The kid is playing with the cat"]
generated_text = ["kid playing the cat"]

results = rouge.compute(predictions=generated_text, references=reference_text)

rouge_l = float(round(results["rougeL"], 2))
print("rouge_l score: ", rouge_l)

Let's define a function that transforms the model predictions and the expected output into the required format

In [ ]:
def get_predictions(example):
    instruction = "You are a knowledgeable assistant. Answer this question truthfully!"

    question = example["question"]

    # Format the input the same way as during training
    prompt = (
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{question}\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"])

    predicted_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # remove the prompt from the response
    predicted_output = predicted_output[len(prompt):]

    return {
        "question": example["question"],
        "predicted_output": predicted_output,
        "correct_output": example["answer"],
    }

For computational reasons, we only evaluate the model on a small subset of the test data.

In [ ]:
test_ds = test_dataset.select(range(30))

In [ ]:
predictions_dataset = test_ds.map(get_predictions, batched=False, remove_columns=test_ds.column_names)
predictions_dataset[2]

In [ ]:
predictions = predictions_dataset["predicted_output"]
references = predictions_dataset["correct_output"]

results = rouge.compute(predictions=predictions, references=references)

print(f"ROUGE-L (F1): {results['rougeL']:.2%}")

## 7) Comparison to untrained SmolLM2 model (not fine-tuned)

In [ ]:
# reload SmolLM2 base model
model_name = "HuggingFaceTB/SmolLM2-135M"
untrained_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

In [ ]:
def get_untrained_predictions(example):
    instruction = "You are a knowledgeable assistant. Answer this question truthfully!"

    question = example["question"]

    # Format the input the same way as during training
    prompt = (
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{question}\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate
    model.eval()
    with torch.no_grad():
      outputs = untrained_model.generate(
          input_ids=inputs["input_ids"],
          attention_mask=inputs["attention_mask"],
          pad_token_id=tokenizer.eos_token_id,
          repetition_penalty=1.2,
      )

    predicted_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # remove the prompt from the response
    predicted_output = predicted_output[len(prompt):]

    return {
        "question": example["question"],
        "predicted_output": predicted_output,
        "correct_output": example["answer"],
    }

In [ ]:
untrained_predictions_dataset = test_ds.map(get_untrained_predictions, batched=False, remove_columns=test_ds.column_names)
untrained_predictions_dataset[2]

In [ ]:
predictions = untrained_predictions_dataset["predicted_output"]
references = untrained_predictions_dataset["correct_output"]

results = rouge.compute(predictions=predictions, references=references)

print(f"ROUGE-L (F1): {results['rougeL']:.2%}")

ROUGE-L (F1) of fine-tuned model: > 70% :)